- preface_text.json : preface text for each scenario
- categorized_lists.json : list of semantically related words (n3, n5, n7, n10)
- random_lists.json : list of arbitrary words (n3, n5, n7, n10)
- intervening_texts.json : key for each scenario. Then key for different length (1,2,3,4,5)

In [20]:
import sys
import os

# Adjust the path to where the src folder is located
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

In [28]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from language_models.dictionary_corpus import Dictionary
from collections import defaultdict



## 1. verify if words from lists are in vocab

In [10]:
# File paths
vocab_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data/vocab.txt"
nouns_path = "/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/nouns_categorized.txt"

# Load vocabulary
with open(vocab_path, "r", encoding="utf-8") as f:
    vocab = set(word.strip() for word in f)

# Load noun words
with open(nouns_path, "r", encoding="utf-8") as f:
    nouns = [word.strip().lower() for word in f if word.strip()]

# Find missing words
missing_words = [word for word in nouns if word not in vocab]

# Print or save
print(f"{len(missing_words)} words not found in vocab:")
for word in missing_words:
    print(word)


389 words not found in vocab:
dormer
alcove
livingroom
rowboat
motorboat
cargoship
clippership
outrigger
speedboat
runabout
hydrofoil
skiff
catamaran
canary
parakeet
blackbird
wren
oriole
starling
titmouse
bluejay
finch
buzzard
flamingo
lark
tangelo
tangerine
blackberry
prunes
cantaloupe
blueberry
boysenberry
raisin
nectarine
kumquat
papaya
honeydew
pliers
sandpaper
awl
sander
plumb
vise
nailset
crowbar
hatchet
ripsaw
drillpress
hacksaw
rasp
jigsaw
sleet
chinook
duststorm
drizzle
breezy
cloudburst
krypton
rubidium
cesium
john
jim
tom
joe
dick
mike
george
harry
steve
larry
paul
sam
dave
fred
charles
jerry
pete
rick
elmer
bruce
gary
karl
henry
ron
jeff
ralph
entomology
bacteriology
biophysics
embryology
agronomy
chicago
helsinki
toronto
london
paris
miami
boston
philadelphia
detroit
dallas
rome
bombay
cleveland
pittsburgh
denver
montreal
tokyo
moscow
atlanta
berlin
stockholm
jerusalem
cairo
vancouver
reno
madrid
memphis
shanghai
copenhagen
naples
brussels
venice
kazoo
xylophone
celesta
b

Those words have already been filtered so they're not in the json

In [13]:
import json

# File paths
json_path = "/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/random_lists.json"
vocab_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data/vocab.txt"

# Load vocab as a lowercase set
with open(vocab_path, "r", encoding="utf-8") as f:
    vocab = set(word.strip() for word in f)

# Load JSON data
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

missing_words = set()
for key, lists in data.items():
    for word_list in lists:
        for word in word_list:
            if word.lower() not in vocab:
                missing_words.add(word.lower())

print(f"{len(missing_words)} lowercase words not in vocab:")
for word in sorted(missing_words):
    print(word)

# Filter lists
filtered_data = {}
for key, lists in data.items():
    filtered_data[key] = [
        word_list for word_list in lists
        if all(word.lower() in vocab for word in word_list)
    ]

# (Optional) Save cleaned data back to file
# cleaned_path = "/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/categorized_lists_cleaned.json"
# with open(cleaned_path, "w", encoding="utf-8") as f:
#     json.dump(filtered_data, f, indent=2)

# print(f"Cleaned data saved to: {cleaned_path}")


0 lowercase words not in vocab:


# Now create test set

In [29]:
class WMTestDataset(Dataset):
    def __init__(self, sentences_path, markers_path, dictionary):
        """
        sentences_path: path to the sentence file (one sentence per line)
        markers_path: path to marker file (CSV/TSV with columns: markers, stimid, list_len, prompt_len)
                      markers column contains array-like strings (e.g. '[0,0,1,1,...]')
        dictionary: object with attribute 'word2idx' (dict mapping words to indices), and
                    should contain an "<unk>" key for unknown words.
        """
        self.dictionary = dictionary
        
        # Load sentences
        with open(sentences_path, "r", encoding="utf-8") as f:
            self.sentences = [line.strip().split() for line in f if line.strip()]
        
        # Load markers metadata file with pandas
        self.markers_df = pd.read_csv(markers_path, sep="\t", converters={
            'markers': lambda x: list(map(int, x.strip('[]').split(',')))
        })
        
        # Sanity check: same number of sentences and markers rows
        assert len(self.sentences) == len(self.markers_df), \
            f"Mismatch: {len(self.sentences)} sentences vs {len(self.markers_df)} marker entries"
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        row = self.markers_df.iloc[idx]
        
        markers = row['markers']
        stimid = row['stimid']
        list_len = row['list_len']
        prompt_len = row['prompt_len']
        
        encoded_sentence = [
            self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>"))
            for word in sentence
        ]
        
        list1 = [sentence[i] for i, tag in enumerate(markers) if tag == 1]
        list2 = [sentence[i] for i, tag in enumerate(markers) if tag == 3]
        
        list1_encoded = [
        encoded_sentence[i] for i, tag in enumerate(markers) if tag == 1
        
        ]
        list2_encoded = [
            encoded_sentence[i] for i, tag in enumerate(markers) if tag == 3
        ]
        
        condition = (list_len, prompt_len)
        
        sample = {
            "sentence": sentence,              # list of tokens (words)
            "encoded_sentence": encoded_sentence, 
            'list1_encoded':list1_encoded,
            "list1":list1,
            "list2_encoded": list2_encoded,
            "list2":list2,
            "stimid": stimid,  
            "condition" : condition
            # "list_len": list_len,              # int
            # "prompt_len": prompt_len           # int
        }
        return sample


In [30]:
sentence_path = '/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/rnn_input_files/categorized_lists_sce1_control.txt'
marker_path = '/scratch2/mrenaudin/colorlessgreenRNNs/wm_tests/rnn_input_files/categorized_lists_sce1_control_markers.txt'
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"
dictionary = Dictionary(data_path)
wm_dataset = WMTestDataset(sentence_path, marker_path, dictionary)

In [26]:
def create_cond_dataloader(dataset):
    """"
    For each conditions (same list len and intervening text len) sentences are the same length, so we can batch by condition. 
    To do so we do one dataloader per conditon. The max batch size is 230 (1 batch per condition)
    """
    condition_loaders = {}
    for (list_len, prompt_len) in dataset.conditions:  # set of tuples
        condition_dataset = [
            sample for sample in dataset
            if sample["list_len"] == list_len and sample["prompt_len"] == prompt_len
        ]
        loader = DataLoader(condition_dataset, batch_size=230, shuffle=False)
        condition_loaders[(list_len, prompt_len)] = loader
    
    return condition_loaders

In [41]:
def collate_fn(batch):
    """Custom collate function to properly handle sentences as lists of strings."""
    sentences = [item["sentence"] for item in batch] 
    list1 = [item["list1"] for item in batch] 
    list2 = [item["list2"] for item in batch] 
    encoded_sentences = torch.stack([torch.tensor(item["encoded_sentence"]) for item in batch])  
    list1_encoded = torch.stack([torch.tensor(item["list1_encoded"]) for item in batch])
    list2_encoded = torch.stack([torch.tensor(item["list2_encoded"]) for item in batch])

    conditions = [item["condition"] for item in batch]

    return {
        "sentence": sentences,
        "encoded_sentence": encoded_sentences,
        "list1": list1,
        "list2": list2,
        "list1_encoded":list1_encoded,
        "list2_encoded":list2_encoded,
        "condition": conditions,
    }


In [42]:
"""
A simpler approach is to use a normal dataloader with a batch size of 230 (or any n divisible by 230)
"""

dataloader = DataLoader(wm_dataset, batch_size=230, collate_fn=collate_fn)


In [ ]:
def compute_seq_nll

In [ ]:
#def eval(model, test_dataloader):
condition_accuracies = defaultdict(int)
condition_counts = defaultdict(int)
correct_pred = 0
sentence_details = []
model.eval()
# Forward pass with hidden state update word by word
with torch.no_grad():
    for batch in dataloader:
        out = None
        sentence = batch["sentence"]
        encoded_sentence = batch["encoded_sentence"]
        list1 = batch["list1"]
        list1_encoded = batch['list1_encoded']
        list2 = batch["list2"]
        list2_encoded = batch['list2_encoded']
        condition = batch["condition"]
        batch_size = sentence.size(0)
        
        surprisal_list1=[]
        
        for idx, token in enumerate(list1):
            if idx>0:
                cache = model.init_cache(batch_size)
                out, cache = model(sent, cache, batch_size)
                
                
                
    #         sent = sentence[:, :5].transpose(0, 1)
    #         cache = model.init_cache(sent,1)  # regarder si on peut mettre du priming
    #         # for i in range(sent.shape[1]):
    #         out, cache = model(sent, cache, 1, replace=None)
    #         log_probs = torch.nn.functional.log_softmax(
    #             out, dim=-1
    #         )  # s(out.squeeze(0))
    #         # déja sur correct et wrong log probs, pas les même résultats que sur extract_predictions.py
    #         correct_log_probs = log_probs[
    #             -1, torch.arange(batch_size), correct
    #         ]  # Shape: [512]
    #         wrong_log_probs = log_probs[-1, torch.arange(batch_size), wrong]
    #         correct_predictions = correct_log_probs >= wrong_log_probs

    #         for i in range(batch_size):
    #             cond = condition[i]
    #             pred = correct_predictions[i].item()  # Convert tensor to Python boolean
    #             condition_counts[cond] += 1
    #             condition_accuracies[cond] += pred

    #             sentence_details.append(
    #                 {
    #                     "sentence": written[i],
    #                     "condition": condition[i],
    #                     "correct_log_prob": correct_log_probs[i],
    #                     "wrong_log_prob": wrong_log_probs[i],
    #                     "model_prefers_correct": pred,
    #                 }
    #             )

    # final_accuracies = {
    #     cond: condition_accuracies[cond] / condition_counts[cond]
    #     for cond in condition_accuracies
    # }
    # return final_accuracies

sentence ['Before', 'the', 'meeting', ',', 'Mary', 'wrote', 'down', 'the', 'following', 'list', 'of', 'words', ':', 'window', ',', 'door', ',', 'roof', '.', 'After', 'the', 'meeting', ',', 'she', 'took', 'a', 'break', 'and', 'had', 'a', 'cup', 'of', 'coffee', '.', 'When', 'she', 'got', 'back', ',', 'she', 'read', 'the', 'list', 'again', ':', 'sparrow', ',', 'heron', ',', 'eagle', '.']
encoded_sentence tensor([ 1464,     3,   115,    11, 12219,  1660,  1100,     3,  1304,  2881,
           53,  2659,     8,   760,    11,  7002,    11,  3589,    18,    20,
            3,   115,    11,     6,   212,    42,  7478,    15,    68,    42,
        14571,    53,  5471,    18,  2532,     6,  1669,  2128,    11,     6,
         1992,     3,  2881,  1784,     8,  3148,    11, 47936,    11, 12199,
           18])
list1 ['window', ',', 'door', ',', 'roof', '.']
list1_encoded tensor([ 760,   11, 7002,   11, 3589,   18])
list2 ['sparrow', ',', 'heron', ',', 'eagle', '.']
list2_encoded tensor([ 3148,   

AttributeError: 'list' object has no attribute 'size'